In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
import dtale

import gc



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)
#the setup (we encapsulated that in a function for keep it constante during the joining to analize purely the gains of each table)
cv , hiperparams = get_baseline_setup() 



In [ ]:

#lets try with the main table without any treatment in the data.
application_train_df = pd.read_csv(cfg.RAW_DATA_DIR / "application_train.csv")

#minimun preparations necessary to be able to train the model with application_train
Y= application_train_df["TARGET"]
X= application_train_df.drop(columns=["TARGET"])
X.drop(columns=["SK_ID_CURR"],inplace=True)
X= cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline")
#0.744 OOF auc is our baseline.

#cleaning memory
del application_train_df,X,Y  
gc.collect()

In [ ]:
#now let's repeat the set up with the version of the silver layer (Cleaned application_train)
#Therefore, this part need execute make_dataset first to generate the fold 01_cleaned

cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
X,Y = prepare_columns(cleaned_application_train)
X= cast_object_into_categoricals(X)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data")
#0.745 OOF auc. Just cleaning the data give us +0.1%, and with less risk of overfitting.


#cleaning memory
del cleaned_application_train,X,Y  
gc.collect()

In [3]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
dtale.show(cleaned_application_train[:100])

2026-06-07 18:23:54,804 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

In [5]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
cleaned_application_train["ratio_debt_income"]= cleaned_application_train["amt_income_total"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_good_credit"]= cleaned_application_train["amt_goods_price"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_annuity_income"] = cleaned_application_train["amt_annuity"] / cleaned_application_train["amt_income_total"]
cleaned_application_train["ratio_days_employed_days_lived"]= cleaned_application_train["days_employed"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["credit_duration"]= cleaned_application_train["amt_credit"] / cleaned_application_train["amt_annuity"]
cleaned_application_train["ext_1_x_2"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_2"]
cleaned_application_train["ext_2_x_3"] = cleaned_application_train["ext_source_2"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["ext_1_x_3"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["amount_of_scores_in_missing"] = cleaned_application_train["ext_source_1_is_missing"] + cleaned_application_train["ext_source_2_is_missing"] + cleaned_application_train["ext_source_3_is_missing"]
X,Y = prepare_columns(cleaned_application_train)
X= cast_object_into_categoricals(X)

cleaned_application_train.to_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")

In [3]:
#now let's repeat the set up with the version of the silver layer (Cleaned application_train)
#Therefore, this part need execute make_dataset first to generate the fold 01_cleaned

cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")

cleaned_application_train["ratio_debt_income"]= cleaned_application_train["amt_income_total"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_debt_age"]= cleaned_application_train["amt_credit"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["ratio_good_credit"]= cleaned_application_train["amt_goods_price"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_annuity_income"] = cleaned_application_train["amt_annuity"] / cleaned_application_train["amt_income_total"]
cleaned_application_train["credit_duration"]= cleaned_application_train["amt_credit"] / cleaned_application_train["amt_annuity"]
cleaned_application_train["ext_1_x_2"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_2"]
cleaned_application_train["ext_2_x_3"] = cleaned_application_train["ext_source_2"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["ext_1_x_3"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["amount_of_scores_in_missing"] = cleaned_application_train["ext_source_1_is_missing"] + cleaned_application_train["ext_source_2_is_missing"] + cleaned_application_train["ext_source_3_is_missing"]
X,Y = prepare_columns(cleaned_application_train)
X= cast_object_into_categoricals(X)

cleaned_application_train.to_parquet(cfg.PROCESSED_DIR / "application_train.train-processed.parquet")

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data")
#0.753 OOF auc with basic feature engineering


#cleaning memory
del cleaned_application_train,X,Y  
gc.collect()

[0]	validation_0-auc:0.71086
[1]	validation_0-auc:0.72363
[2]	validation_0-auc:0.72832
[3]	validation_0-auc:0.73179
[4]	validation_0-auc:0.73419
[5]	validation_0-auc:0.73692
[6]	validation_0-auc:0.73913
[7]	validation_0-auc:0.74201
[8]	validation_0-auc:0.74392
[9]	validation_0-auc:0.74571
[10]	validation_0-auc:0.74782
[11]	validation_0-auc:0.74835
[12]	validation_0-auc:0.74945
[13]	validation_0-auc:0.74958
[14]	validation_0-auc:0.75064
[15]	validation_0-auc:0.75212
[16]	validation_0-auc:0.75237
[17]	validation_0-auc:0.75269
[18]	validation_0-auc:0.75289
[19]	validation_0-auc:0.75307
[20]	validation_0-auc:0.75360
[21]	validation_0-auc:0.75364
[22]	validation_0-auc:0.75408
[23]	validation_0-auc:0.75423
[24]	validation_0-auc:0.75427
[25]	validation_0-auc:0.75402
[26]	validation_0-auc:0.75381
[27]	validation_0-auc:0.75437
[28]	validation_0-auc:0.75455
[29]	validation_0-auc:0.75496
[30]	validation_0-auc:0.75455
[31]	validation_0-auc:0.75527
[32]	validation_0-auc:0.75513
[33]	validation_0-au

258